In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import HuggingFaceEndpoint
from typing_extensions import TypedDict
from dotenv import load_dotenv
import os

In [11]:
load_dotenv()

True

In [12]:
model = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-v0.1",
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN"),
    temperature=0.7,
    max_new_tokens=100
)

In [13]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str

In [14]:
def create_outline(state: BlogState) -> BlogState:

    #fetch title
    title = state['title']

    #call LLM gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt)

    #update state
    state['outline'] = outline

    return state

In [15]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'write a detailed blog on the title - {title} using the following outline \n {outline}'

    content = model.invoke(prompt)

    state['content'] = content

    return state

In [16]:
graph = StateGraph(BlogState)

#nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

#edges

graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog',END)

workflow = graph.compile()

In [17]:
initial_state = {'title': 'Rise of AI in india'}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Rise of AI in india', 'outline': ', 2023: Opportunities and Challenges\n\n1. Introduction: Provide a brief overview of the blog topic, including the significance of AI in India and its current state.\n\n2. Rise of AI in India: Discuss the increasing adoption of AI in various industries and sectors in India, highlighting specific use cases and examples.\n\n3. Opportunities: Analyze the potential benefits and opportunities that AI brings to India, including job', 'content': " creation, economic growth, and improved efficiency in various sectors.\n\n4. Challenges: Identify and discuss the challenges and obstacles that India faces in fully realizing the potential of AI, such as lack of skilled workforce, regulatory frameworks, and data privacy concerns.\n\n5. Government Initiatives: Analyze the government's efforts in promoting AI development in India, including policies, funding programs, and initiatives to address the challenges.\n\n6. Private S"}


In [19]:
print(final_state['outline'])

, 2023: Opportunities and Challenges

1. Introduction: Provide a brief overview of the blog topic, including the significance of AI in India and its current state.

2. Rise of AI in India: Discuss the increasing adoption of AI in various industries and sectors in India, highlighting specific use cases and examples.

3. Opportunities: Analyze the potential benefits and opportunities that AI brings to India, including job
